In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Cell 1 — define the DummyJSON user payload schema

In [0]:
dj_user_payload_schema = StructType([
    StructField("id", IntegerType()),
    StructField("firstName", StringType()),
    StructField("lastName", StringType()),
    StructField("email", StringType()),
    StructField("phone", StringType()),
    StructField("username", StringType()),
    StructField("birthDate", StringType()),
    StructField("address", StructType([
        StructField("address", StringType()),
        StructField("city", StringType()),
        StructField("state", StringType()),
        StructField("postalCode", StringType()),
        StructField("country", StringType()),
    ])),
    StructField("company", StructType([
        StructField("department", StringType()),
        StructField("name", StringType()),
        StructField("title", StringType()),
    ])),
])

### Cell 2 — create silver schema table for customers, CDF enabled

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS mia_catalog.silver.silver_customers (
        customer_key STRING,
        first_name STRING,
        last_name STRING,
        email STRING,
        phone STRING,
        username STRING,
        birth_date STRING,
        address STRING,
        city STRING,
        state STRING,
        postal_code STRING,
        country STRING,
        company_name STRING,
        company_department STRING,
        job_title STRING,
        record_hash STRING,
        last_updated_ts TIMESTAMP
    )
    USING DELTA
    TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
print("mia_catalog.silver.silver_customers ready, CDF enabled")

mia_catalog.silver.silver_customers ready, CDF enabled


### Cell 3 — parse + flatten DummyJSON users

In [0]:
dj_users_parsed = (
    spark.table("bronze_dummyjson_users")
    .withColumn("parsed", from_json(col("raw_payload"), dj_user_payload_schema))
    .select(
        col("parsed.id").cast("string").alias("customer_key"),
        col("parsed.firstName").alias("first_name"),
        col("parsed.lastName").alias("last_name"),
        col("parsed.email").alias("email"),
        col("parsed.phone").alias("phone"),
        col("parsed.username").alias("username"),
        col("parsed.birthDate").alias("birth_date"),
        col("parsed.address.address").alias("address"),
        col("parsed.address.city").alias("city"),
        col("parsed.address.state").alias("state"),
        col("parsed.address.postalCode").alias("postal_code"),
        col("parsed.address.country").alias("country"),
        col("parsed.company.name").alias("company_name"),
        col("parsed.company.department").alias("company_department"),
        col("parsed.company.title").alias("job_title"),
    )
)

display(dj_users_parsed.limit(5))

customer_key,first_name,last_name,email,phone,username,birth_date,address,city,state,postal_code,country,company_name,company_department,job_title
1,Emily,Johnson,emily.johnson@x.dummyjson.com,+81 965-431-3024,emilys,1996-5-30,626 Main Street,Phoenix,Mississippi,29112,United States,"Dooley, Kozey and Cronin",Engineering,Sales Manager
2,Michael,Williams,michael.williams@x.dummyjson.com,+49 258-627-6644,michaelw,1989-8-10,385 Fifth Street,Houston,Alabama,38807,United States,Spinka - Dickinson,Support,Support Specialist
3,Sophia,Brown,sophia.brown@x.dummyjson.com,+81 210-652-2785,sophiab,1982-11-6,1642 Ninth Street,Washington,Alabama,32822,United States,Schiller - Zieme,Research and Development,Accountant
4,James,Davis,james.davis@x.dummyjson.com,+49 614-958-9364,jamesd,1979-5-4,238 Jefferson Street,Seattle,Pennsylvania,68354,United States,Pagac and Sons,Support,Research Analyst
5,Emma,Miller,emma.miller@x.dummyjson.com,+91 759-776-1614,emmaj,1994-6-13,607 Fourth Street,Jacksonville,Colorado,26593,United States,Graham - Gulgowski,Human Resources,Quality Assurance Engineer


### Cell 4 — compute record_hash + timestamp

In [0]:
silver_customers_with_hash = (
    dj_users_parsed
    .withColumn(
        "record_hash",
        md5(concat_ws("|",
            col("first_name"), col("last_name"), col("email"), col("phone"),
            col("address"), col("city"), col("state"), col("postal_code"),
            col("country"), col("company_name"), col("company_department"), col("job_title")
        ))
    )
    .withColumn("last_updated_ts", current_timestamp())
)

### Cell 5a — register the DataFrame as a temp view

In [0]:
silver_customers_with_hash.createOrReplaceTempView("silver_customers_updates")

In [0]:
spark.sql("""
    MERGE INTO mia_catalog.silver.silver_customers AS target
    USING silver_customers_updates AS source
    ON target.customer_key = source.customer_key
    WHEN MATCHED AND target.record_hash != source.record_hash THEN
        UPDATE SET *
    WHEN NOT MATCHED THEN
        INSERT *
""")

print("MERGE completed")

MERGE completed


In [0]:
display(spark.sql("SELECT count(*) FROM mia_catalog.silver.silver_customers"))

count(*)
208
